# Credit Card Fraud EDA

Task 1 notebook for anonymized bank transaction cleaning checks, distributions, target imbalance, and amount/time patterns.

In [3]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve().parents[0] if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.config import CREDITCARD
from src.data_utils import basic_clean, class_distribution, load_creditcard
from src.features import engineer_creditcard_features


In [4]:
credit_raw = load_creditcard(CREDITCARD)
credit_raw.head()

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0


In [5]:
cleaning_summary = {
    'rows_before': len(credit_raw),
    'duplicates': int(credit_raw.duplicated().sum()),
    'missing_values': credit_raw.isna().sum().to_dict(),
}
cleaning_summary

{'rows_before': 284807,
 'duplicates': 1081,
 'missing_values': {'Time': 0,
  'V1': 0,
  'V2': 0,
  'V3': 0,
  'V4': 0,
  'V5': 0,
  'V6': 0,
  'V7': 0,
  'V8': 0,
  'V9': 0,
  'V10': 0,
  'V11': 0,
  'V12': 0,
  'V13': 0,
  'V14': 0,
  'V15': 0,
  'V16': 0,
  'V17': 0,
  'V18': 0,
  'V19': 0,
  'V20': 0,
  'V21': 0,
  'V22': 0,
  'V23': 0,
  'V24': 0,
  'V25': 0,
  'V26': 0,
  'V27': 0,
  'V28': 0,
  'Amount': 0,
  'Class': 0}}

In [6]:
credit = engineer_creditcard_features(basic_clean(credit_raw))
class_distribution(credit, 'Class')

,Class,count,percentage
0,0,283253,99.83329
1,1,473,0.16671


In [7]:
credit[['Time', 'Amount', 'amount_log1p', 'hour_of_day']].describe()

,Time,Amount,amount_log1p,hour_of_day
count,283726.000000,283726.000000,283726.000000,283726.000000
mean,94811.077600,88.472687,3.153760,14.045646
std,47481.047891,250.399437,1.657080,5.834817
min,0.000000,0.000000,0.000000,0.000000
25%,54204.750000,5.600000,1.887070,10.000000
50%,84692.500000,22.000000,3.135494,15.000000
75%,139298.000000,77.510000,4.363226,19.000000
max,172792.000000,25691.160000,10.153941,23.000000


In [8]:
credit.groupby('Class')[['Amount', 'amount_log1p', 'hour_of_day']].agg(['count', 'median', 'mean'])

Amount                    amount_log1p                     hour_of_day  \
        count median        mean        count    median      mean       count   
Class                                                                           
0      283253  22.00   88.413575       283253  3.135494  3.154289      283253   
1         473   9.82  123.871860          473  2.381396  2.837549         473   

                         
      median       mean  
Class                    
0       15.0  14.049638  
1       12.0  11.655391

In [9]:
credit.groupby(['hour_of_day', 'Class']).size().unstack(fill_value=0).assign(
    fraud_rate=lambda x: x.get(1, 0) / x.sum(axis=1)
).sort_values('fraud_rate', ascending=False).head(10)

Class,0,1,fraud_rate
hour_of_day,,,
2.0,3260,48,0.014510
4.0,2181,23,0.010436
3.0,3470,17,0.004875
5.0,2977,11,0.003681
7.0,7210,23,0.003180
11.0,16728,53,0.003158
1.0,4198,10,0.002376
6.0,4073,9,0.002205
17.0,16102,28,0.001736


Use these outputs to document why accuracy is misleading and why AUC-PR, F1, and confusion matrix should drive later model evaluation.